# Minimal educational BLIP-2: frozen CLIP ViT-B/32 + Q-Former + frozen GPT-2.
Stage 1: ITC + ITM + ITG on the Q-Former (one set of shared self-attention
         weights, three attention masks).
         
Stage 2: project query outputs into GPT-2 embedding space, prepend them to the
         caption embeddings, train with the LM loss (GPT-2 frozen).


In [ ]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
from transformers import CLIPVisionModel, CLIPImageProcessor, GPT2LMHeadModel, GPT2Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
VISION_NAME = "openai/clip-vit-base-patch32"
LLM_NAME = "gpt2"

NUM_QUERIES = 32
HIDDEN = 768
NUM_LAYERS = 2
NUM_HEADS = 8
MAX_TEXT_LEN = 24
BATCH_SIZE = 16
STAGE1_EPOCHS = 8
STAGE2_EPOCHS = 8
LR = 1e-4
NUM_TRAIN = 512  # number of (image, caption) pairs used

# ----------------------------------------------------------------------------
# 1. Data: try a small HF captioning dataset, else synthesize
# ----------------------------------------------------------------------------
COLORS = {"red": (220, 40, 40), "green": (40, 180, 60), "blue": (40, 80, 220),
          "yellow": (230, 210, 40), "purple": (150, 60, 190), "orange": (240, 140, 30)}
SHAPES = ["circle", "square", "triangle"]


def make_synthetic_sample():
    color = random.choice(list(COLORS))
    shape = random.choice(SHAPES)
    img = Image.new("RGB", (224, 224), (245, 245, 245))
    d = ImageDraw.Draw(img)
    cx, cy = random.randint(80, 144), random.randint(80, 144)
    r = random.randint(35, 60)
    c = COLORS[color]
    if shape == "circle":
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=c)
    elif shape == "square":
        d.rectangle([cx - r, cy - r, cx + r, cy + r], fill=c)
    else:
        d.polygon([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], fill=c)
    return img, f"a {color} {shape}"


def load_pairs(n):
    try:
        from datasets import load_dataset
        # Best-guess small captioning dataset; falls back to synthetic on any failure.
        ds = load_dataset("nlphuji/flickr30k", split=f"test[:{n}]")
        pairs = []
        for ex in ds:
            cap = ex["caption"][0] if isinstance(ex["caption"], list) else ex["caption"]
            pairs.append((ex["image"].convert("RGB"), cap.lower().strip()))
        print(f"Loaded {len(pairs)} real pairs.")
        return pairs
    except Exception as e:
        print(f"Real dataset unavailable ({type(e).__name__}); using synthetic data.")
        return [make_synthetic_sample() for _ in range(n)]


class PairDataset(Dataset):
    def __init__(self, pairs, processor, tokenizer):
        self.pairs, self.processor, self.tok = pairs, processor, tokenizer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        img, cap = self.pairs[i]
        pix = self.processor(images=img, return_tensors="pt")["pixel_values"][0]
        ids = self.tok(cap, truncation=True, max_length=MAX_TEXT_LEN - 1,
                       padding="max_length", return_tensors="pt")
        return pix, ids["input_ids"][0], ids["attention_mask"][0]


# ----------------------------------------------------------------------------
# 2. Q-Former: shared self-attention over [queries ; text], cross-attention
#    to image features for query positions only.
# ----------------------------------------------------------------------------
class QFormerBlock(nn.Module):
    def __init__(self, dim, heads, has_cross):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ln1 = nn.LayerNorm(dim)
        self.has_cross = has_cross
        if has_cross:
            self.cross_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
            self.ln_cross = nn.LayerNorm(dim)
        # separate FFNs for query and text positions
        self.ffn_q = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
        self.ffn_t = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
        self.ln2_q, self.ln2_t = nn.LayerNorm(dim), nn.LayerNorm(dim)

    def forward(self, x, nq, img_feats, attn_mask, key_padding_mask):
        # attn_mask: [L, L] bool, True = blocked. key_padding_mask: [B, L] bool, True = pad.
        a, _ = self.self_attn(x, x, x, attn_mask=attn_mask, key_padding_mask=key_padding_mask)
        x = self.ln1(x + a)
        q, t = x[:, :nq], x[:, nq:]
        if self.has_cross:  # only queries see the image
            c, _ = self.cross_attn(q, img_feats, img_feats)
            q = self.ln_cross(q + c)
        q = self.ln2_q(q + self.ffn_q(q))
        if t.size(1) > 0:
            t = self.ln2_t(t + self.ffn_t(t))
        return torch.cat([q, t], dim=1)


class QFormer(nn.Module):
    def __init__(self, vocab_size, img_dim):
        super().__init__()
        self.nq = NUM_QUERIES
        self.queries = nn.Parameter(torch.randn(1, NUM_QUERIES, HIDDEN) * 0.02)
        self.img_proj = nn.Linear(img_dim, HIDDEN)
        self.tok_emb = nn.Embedding(vocab_size, HIDDEN)
        self.pos_emb = nn.Embedding(MAX_TEXT_LEN, HIDDEN)
        # cross-attention in every other block (blocks 0, 2, ...)
        self.blocks = nn.ModuleList([QFormerBlock(HIDDEN, NUM_HEADS, has_cross=(i % 2 == 0))
                                     for i in range(NUM_LAYERS)])

    def build_mask(self, mode, text_len):
        L = self.nq + text_len
        blocked = torch.zeros(L, L, dtype=torch.bool, device=device)
        nq = self.nq
        if mode == "itc":      # unimodal: queries<->queries, text<->text only
            blocked[:nq, nq:] = True
            blocked[nq:, :nq] = True
        elif mode == "itm":    # bidirectional: everything sees everything
            pass
        elif mode == "itg":    # queries blind to text; text sees queries + earlier text
            blocked[:nq, nq:] = True
            causal = torch.triu(torch.ones(text_len, text_len, dtype=torch.bool, device=device), 1)
            blocked[nq:, nq:] = causal
        return blocked

    def forward(self, img_feats, input_ids=None, text_pad=None, mode="itm"):
        B = img_feats.size(0)
        img = self.img_proj(img_feats)
        q = self.queries.expand(B, -1, -1)
        if input_ids is not None:
            pos = torch.arange(input_ids.size(1), device=device)
            t = self.tok_emb(input_ids) + self.pos_emb(pos)[None]
            x = torch.cat([q, t], dim=1)
            mask = self.build_mask(mode, input_ids.size(1))
            kpm = torch.cat([torch.zeros(B, self.nq, dtype=torch.bool, device=device), text_pad], dim=1)
        else:
            x, mask, kpm = q, None, None
        for blk in self.blocks:
            x = blk(x, self.nq, img, mask, kpm)
        return x[:, :self.nq], x[:, self.nq:]


# ----------------------------------------------------------------------------
# 3. Stage 1 wrapper: ITC + ITM + ITG
# ----------------------------------------------------------------------------
class Stage1(nn.Module):
    def __init__(self, qformer, vocab_size, embed_dim=256):
        super().__init__()
        self.qf = qformer
        self.img_head = nn.Linear(HIDDEN, embed_dim)
        self.txt_head = nn.Linear(HIDDEN, embed_dim)
        self.itm_head = nn.Linear(HIDDEN, 2)
        self.lm_head = nn.Linear(HIDDEN, vocab_size)
        self.temp = nn.Parameter(torch.tensor(0.07))

    @staticmethod
    def pool_text(t, pad):  # mean over non-pad tokens
        m = (~pad).float().unsqueeze(-1)
        return (t * m).sum(1) / m.sum(1).clamp(min=1)

    def forward(self, img_feats, ids, pad):
        B = ids.size(0)
        # ---- ITC (unimodal mask)
        q_out, t_out = self.qf(img_feats, ids, pad, mode="itc")
        iv = F.normalize(self.img_head(q_out), dim=-1)                 # [B, nq, d]
        tv = F.normalize(self.txt_head(self.pool_text(t_out, pad)), dim=-1)  # [B, d]
        sim = torch.einsum("iqd,td->itq", iv, tv).max(-1).values / self.temp.clamp(min=0.01)  # [B_img, B_txt]
        tgt = torch.arange(B, device=device)
        loss_itc = (F.cross_entropy(sim, tgt) + F.cross_entropy(sim.t(), tgt)) / 2

        # ---- ITM (bidirectional mask) with hard negatives from ITC similarity
        with torch.no_grad():
            w_i2t = F.softmax(sim, dim=1) + 1e-4
            w_i2t.fill_diagonal_(0)
            w_t2i = F.softmax(sim.t(), dim=1) + 1e-4
            w_t2i.fill_diagonal_(0)
            neg_txt = torch.multinomial(w_i2t, 1).squeeze(1)   # hard text for each image
            neg_img = torch.multinomial(w_t2i, 1).squeeze(1)   # hard image for each text
        img_all = torch.cat([img_feats, img_feats, img_feats[neg_img]], 0)
        ids_all = torch.cat([ids, ids[neg_txt], ids], 0)
        pad_all = torch.cat([pad, pad[neg_txt], pad], 0)
        labels = torch.cat([torch.ones(B), torch.zeros(2 * B)]).long().to(device)
        q_itm, _ = self.qf(img_all, ids_all, pad_all, mode="itm")
        logits = self.itm_head(q_itm).mean(1)                  # average per-query logits
        loss_itm = F.cross_entropy(logits, labels)

        # ---- ITG (multimodal causal mask), next-token prediction
        _, t_g = self.qf(img_feats, ids, pad, mode="itg")
        pred = self.lm_head(t_g[:, :-1])
        target = ids[:, 1:].masked_fill(pad[:, 1:], -100)
        loss_itg = F.cross_entropy(pred.reshape(-1, pred.size(-1)), target.reshape(-1), ignore_index=-100)
        return loss_itc, loss_itm, loss_itg


# ----------------------------------------------------------------------------
# 4. Stage 2: Q-Former -> linear projection -> frozen GPT-2
# ----------------------------------------------------------------------------
class Stage2(nn.Module):
    def __init__(self, qformer, llm):
        super().__init__()
        self.qf = qformer
        self.llm = llm
        for p in self.llm.parameters():
            p.requires_grad = False
        self.proj = nn.Linear(HIDDEN, llm.config.n_embd)

    def visual_prefix(self, img_feats):
        q_out, _ = self.qf(img_feats)               # image-only path, no text
        return self.proj(q_out)                     # [B, nq, n_embd]

    def forward(self, img_feats, ids, pad):
        prefix = self.visual_prefix(img_feats)
        tok_emb = self.llm.transformer.wte(ids)
        x = torch.cat([prefix, tok_emb], dim=1)
        attn = torch.cat([torch.ones(ids.size(0), NUM_QUERIES, device=device, dtype=torch.long),
                          (~pad).long()], dim=1)
        labels = torch.cat([torch.full((ids.size(0), NUM_QUERIES), -100, device=device),
                            ids.masked_fill(pad, -100)], dim=1)
        return self.llm(inputs_embeds=x, attention_mask=attn, labels=labels).loss

    @torch.no_grad()
    def generate(self, img_feats, tok, max_new=12):
        prefix = self.visual_prefix(img_feats)
        out = self.llm.generate(inputs_embeds=prefix, max_new_tokens=max_new, do_sample=False,
                                pad_token_id=tok.eos_token_id)
        return tok.batch_decode(out, skip_special_tokens=True)


# ----------------------------------------------------------------------------
# 5. Main
# ----------------------------------------------------------------------------
def main():
    random.seed(0)
    torch.manual_seed(0)

    processor = CLIPImageProcessor.from_pretrained(VISION_NAME)
    vision = CLIPVisionModel.from_pretrained(VISION_NAME).to(device).eval()
    for p in vision.parameters():
        p.requires_grad = False
    tok = GPT2Tokenizer.from_pretrained(LLM_NAME)
    tok.pad_token = tok.eos_token
    llm = GPT2LMHeadModel.from_pretrained(LLM_NAME).to(device).eval()

    pairs = load_pairs(NUM_TRAIN)
    loader = DataLoader(PairDataset(pairs, processor, tok), batch_size=BATCH_SIZE,
                        shuffle=True, drop_last=True)

    @torch.no_grad()
    def encode_image(pix):
        # second-to-last layer features, CLS token dropped
        hs = vision(pixel_values=pix, output_hidden_states=True).hidden_states[-2]
        return hs[:, 1:]

    img_dim = vision.config.hidden_size
    qformer = QFormer(len(tok), img_dim).to(device)

    # ---- Stage 1
    s1 = Stage1(qformer, len(tok)).to(device)
    opt = torch.optim.AdamW(s1.parameters(), lr=LR)
    for ep in range(STAGE1_EPOCHS):
        tot = [0.0, 0.0, 0.0]
        for pix, ids, am in loader:
            pix, ids, pad = pix.to(device), ids.to(device), (am == 0).to(device)
            feats = encode_image(pix)
            l_itc, l_itm, l_itg = s1(feats, ids, pad)
            loss = l_itc + l_itm + l_itg
            opt.zero_grad(); loss.backward(); opt.step()
            for k, v in enumerate((l_itc, l_itm, l_itg)):
                tot[k] += v.item()
        n = len(loader)
        print(f"[Stage1] epoch {ep+1}: ITC {tot[0]/n:.3f} | ITM {tot[1]/n:.3f} | ITG {tot[2]/n:.3f}")

    # ---- Stage 2
    s2 = Stage2(qformer, llm).to(device)
    params = [p for p in s2.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR)
    for ep in range(STAGE2_EPOCHS):
        tot = 0.0
        for pix, ids, am in loader:
            pix, ids, pad = pix.to(device), ids.to(device), (am == 0).to(device)
            loss = s2(encode_image(pix), ids, pad)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()
        print(f"[Stage2] epoch {ep+1}: LM loss {tot/len(loader):.3f}")

    # ---- Qualitative check
    s2.eval()
    for i in range(4):
        img, cap = pairs[i]
        pix = processor(images=img, return_tensors="pt")["pixel_values"].to(device)
        gen = s2.generate(encode_image(pix), tok)[0]
        print(f"GT: {cap!r} | GEN: {gen!r}")


main()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.final_layer_norm.bias                             | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Real dataset unavailable (RuntimeError); using synthetic data.
[Stage1] epoch 1: ITC 2.270 | ITM 0.833 | ITG 4.177
[Stage1] epoch 2: ITC 0.560 | ITM 0.639 | ITG 1.555
[Stage1] epoch 3: ITC 0.543 | ITM 0.627 | ITG 0.385
[Stage1] epoch 4: ITC 0.536 | ITM 0.601 | ITG 0.025
[Stage1] epoch 5: ITC 0.515 | ITM 0.570 | ITG 0.011
[Stage1] epoch 6: ITC 0.489 | ITM 0.553 | ITG 0.008
[Stage1] epoch 7: ITC 0.467 | ITM 0.528 | ITG 0.006


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


[Stage1] epoch 8: ITC 0.497 | ITM 0.550 | ITG 0.005
[Stage2] epoch 1: LM loss 4.084
[Stage2] epoch 2: LM loss 0.182
[Stage2] epoch 3: LM loss 0.010
[Stage2] epoch 4: LM loss 0.004
[Stage2] epoch 5: LM loss 0.003
[Stage2] epoch 6: LM loss 0.002
[Stage2] epoch 7: LM loss 0.002
[Stage2] epoch 8: LM loss 0.001
GT: 'a yellow square' | GEN: 'a yellow square square square square square square square square square square'
GT: 'a yellow square' | GEN: 'a yellow square square square square square square square square square square'
GT: 'a purple circle' | GEN: 'a purple circle." purple circle." purple circle." purple circle'
GT: 'a green circle' | GEN: 'a green circle?" green circle green circle green circle green circle'
